# 01 - Your first Adaptive Data run

**adaption-devkit** is a *community, unofficial* toolkit. It is **not affiliated with or endorsed by Adaption Labs**. License: Apache-2.0. Author: Aivaras Navardauskas (MANIFESTA), GitHub A1VARA5.

This notebook walks an absolute beginner through one full, cheap loop:

1. Load a tiny local dataset.
2. Build the column mapping (the most important step).
3. **Estimate** the run cost first (always do this).
4. Run a small **pilot** capped with `max_rows`.
5. Read the headline number, `improvement_percent`, from the evaluation.

All outputs below are **illustrative** - this notebook was authored, not executed, so the numbers you see are placeholders that show the shape of the response.

## Setup: install and authenticate

Authentication is via **environment variables only**. Never hardcode your host or key into a notebook.

Set these in your shell *before* launching Jupyter:

```bash
export ADAPTION_BASE_URL="https://api.prod.adaptionlabs.ai"   # your Adaption API host
export ADAPTION_API_KEY="pt_live_..."                    # your secret key
```

On Windows PowerShell:

```powershell
$env:ADAPTION_BASE_URL = "https://api.prod.adaptionlabs.ai"
$env:ADAPTION_API_KEY  = "pt_live_..."
```

In [ ]:
# Run once if you do not already have the SDK. The leading '%' makes this a notebook magic.
%pip install -q adaption pandas

In [ ]:
import os
from adaption import Adaption

# The SDK reads ADAPTION_API_KEY from the environment when you omit the argument.
# We pass base_url explicitly from the env so the host is never hardcoded either.
BASE_URL = os.environ["ADAPTION_BASE_URL"]  # raises a clear KeyError if you forgot to set it
assert os.environ.get("ADAPTION_API_KEY"), "Set ADAPTION_API_KEY in your environment first."

client = Adaption(base_url=BASE_URL)  # api_key picked up from ADAPTION_API_KEY
print("Client ready. Authenticated via environment variables.")

## Step 1 - Load and inspect the tiny sample dataset

The sample in `sample_data/sample.csv` follows the **completion only, high quality answers** pattern. There is **no prompt column**: every row is a polished marketing blurb (`completion`) plus two `context` columns (`channel`, `audience`).

**Why completion only?** When you give the platform only high quality answers, it **synthesizes a matching prompt for each row**. That is ideal when you have a curated voice corpus but no instructions written out.

**Read with `utf-8-sig`** so a byte order mark (common in exported CSVs) does not corrupt the first column name.

In [ ]:
import pandas as pd

CSV_PATH = "sample_data/sample.csv"
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")  # utf-8-sig strips a leading BOM if present
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")
df.head(3)

### A word on deduplication and anchors

Deduplication is **always on** and it keys on the **prompt**. That has a sharp consequence:

- Our sample is **completion only**, so every row's *answer* is unique and nothing collapses. Good.
- If instead you mapped a `prompt` column where every row shared the **same templated instruction** (for example `"Write a marketing blurb"`), deduplication would treat them as duplicates and **collapse your dataset to one row**. That silently destroys a run.

**Rule of thumb:** keep anchors unique. Use **completion only** (unique answers) or genuinely distinct prompts. Never feed a constant templated prompt across many rows.

## Step 2 - Upload the file (ingest)

`upload_file` runs create + presigned upload + complete in one call and returns a `dataset_id`. Accepted formats: `.csv .json .jsonl .parquet`.

In [ ]:
uploaded = client.datasets.upload_file(CSV_PATH, name="brandvoice-firstrun-sample")
dataset_id = uploaded.dataset_id
print("dataset_id:", dataset_id)  # illustrative, e.g. 'ds_9f3a...'

# A local upload is ready immediately, but checking status is a good habit.
status = client.datasets.get_status(dataset_id)
print("row_count:", status.row_count)  # not None means ingestion finished

## Step 3 - Build the column mapping

A run needs **at least one anchor**: a `prompt` column **or** a `completion` column.

Our sample is completion only, so:

- `completion` -> the high quality answer column.
- `context` is a **list** of supporting columns. Here we pass both `channel` and `audience`.
- We deliberately do **not** set `prompt`. The platform will synthesize prompts from the completions.

In [ ]:
column_mapping = {
    "completion": "completion",        # the anchor: high quality marketing copy
    "context": ["channel", "audience"],  # context is ALWAYS a list, even for one column
    # no 'prompt' on purpose -> the platform synthesizes a prompt per row
}
column_mapping

## Step 4 - Estimate FIRST (never skip this)

Credits are limited. `estimate=True` validates your mapping and quotes credits + minutes **without starting a run or charging anything**. `run_id` comes back `None` for an estimate.

For a marketing voice corpus, a good starting recipe set (see the recipe-selection guide) is:

- `deduplication: True` - almost always on.
- `prompt_rephrase: True` - more prompt variety since prompts are synthesized.
- a `blueprint` brand control to lock the voice on every completion.

In [ ]:
BLUEPRINT = (
    "Write in a warm, concrete, benefit-led marketing voice. "
    "Lead with the customer outcome, avoid hype and cliche, keep sentences crisp."
)

estimate = client.datasets.run(
    dataset_id,
    column_mapping=column_mapping,
    brand_controls={
        "length": "concise",
        "blueprint": BLUEPRINT,
    },
    recipe_specification={
        "recipes": {
            "deduplication": True,
            "prompt_rephrase": True,
        }
    },
    estimate=True,  # <-- quote only, no run, no charge
)

# Illustrative output shape:
print("estimate:", estimate.estimate)                       # True
print("estimated_credits_consumed:", estimate.estimated_credits_consumed)  # e.g. 6
print("estimated_minutes:", estimate.estimated_minutes)     # e.g. 3
print("run_id:", estimate.run_id)                           # None for an estimate

## Step 5 - Run a small pilot capped with `max_rows`

Once the quote looks sane, run for real but **cap the rows** with `job_specification.max_rows`. Piloting on a slice keeps cost tiny while you confirm the config moves the score. Our sample is only 12 rows, so we cap at all of them; on a real corpus you would start at 200-500.

In [ ]:
run = client.datasets.run(
    dataset_id,
    column_mapping=column_mapping,
    brand_controls={"length": "concise", "blueprint": BLUEPRINT},
    recipe_specification={"recipes": {"deduplication": True, "prompt_rephrase": True}},
    job_specification={
        "max_rows": 12,                       # pilot cap; raise this for the full corpus
        "idempotency_key": "firstrun-pilot-1",  # safe-retry key so retries do not double-charge
    },
    estimate=False,  # this one actually starts the pipeline
)
print("run_id:", run.run_id)  # now a real id, e.g. 'run_4c81...'

In [ ]:
from adaption import DatasetTimeout

# Block until the adaptation run reaches a terminal state.
# NOTE: run completion is NOT the same as evaluation completion (next cell).
try:
    status = client.datasets.wait_for_completion(dataset_id, timeout=1800)
    print("run status:", status.status)  # 'succeeded' | 'failed'
    if status.error:
        print("error:", status.error.message)
except DatasetTimeout as e:
    print("timed out after", e.timeout, "s; last status:", e.last_status)

## Step 6 - Read `improvement_percent` (the headline number)

Evaluation runs on its **own schedule**. A run can be `succeeded` while evaluation is still `pending` or `running`. The improvement number lives in `get_evaluation`, **not** in the run status. So we poll evaluation separately.

In [ ]:
import time

# Poll the evaluation until it reaches a terminal state.
while True:
    ev = client.datasets.get_evaluation(dataset_id)
    if ev.status not in ("pending", "running"):
        break
    print("evaluation", ev.status, "... waiting")
    time.sleep(5)

print("evaluation status:", ev.status)  # 'succeeded' | 'failed' | 'skipped'
if ev.status == "succeeded" and ev.quality:
    print("score_before:", ev.quality.score_before)            # 0-10, illustrative e.g. 6.1
    print("score_after: ", ev.quality.score_after)             # 0-10, illustrative e.g. 7.8
    print("improvement_percent:", ev.quality.improvement_percent)  # <-- the hackathon number

## Recap

- Auth came **only** from `ADAPTION_BASE_URL` and `ADAPTION_API_KEY`.
- We used a **completion only** anchor so deduplication (which keys on the prompt) never collapsed our rows.
- We **estimated before running**, then piloted with `max_rows`.
- We read `improvement_percent` from `get_evaluation`, polled separately from run status.

Next: **02_import_from_hf.ipynb** to pull a dataset straight from Hugging Face, and **03_publish.ipynb** to release the adapted result.